# THGS 2D-Source Ablation — Phase 1: SAM (A) vs FastSAM (B)

**3D 미경유 Pure 2D 품질 + 속도 비교** — Ramen 씬, 7 평가 프레임.

단일 변수: mask proposer (SAM ViT-H 4-scale → FastSAM 4-conf-threshold).
CLIP 인코더(OpenCLIP ViT-B/16, 512D), NMS, mask2segmap(224 crop), cumulative offset 모두 동일.

## 선행 조건
- `THGS_Pipeline_ramen.ipynb` 한 번 실행되어 Drive 캐시(`THGS_cache/ramen_language_features`)가 존재해야 A의 features 복원 가능.

## 셀 구조
| 셀 | 내용 |
|---|---|
| 0 | 환경 설정 (THGS clone + 의존성 + FastSAM) |
| 1 | Drive 캐시에서 A의 `language_features/` 복원 |
| 2 | FastSAM 가중치 다운로드 |
| 3 | B용 인코딩: `scripts/encode_fastsam_clip.py` 실행 |
| 4 | A vs B mask proposer 품질 격리 (CLIP-free, GT 대비 mask coverage) |
| 5 | A 평가 (s_only + best4) |
| 6 | B 평가 (s_only + best4) |
| 7 | Timing 통합 표 |
| 8 | Level별 정성 시각화 (A grid, B grid) |
| 9 | A vs B 비교 시각화 (선별 케이스) |
| 10 | 종합 차트 + summary.md |

---
## Cell 0. 환경 설정

In [ ]:
# GPU 확인 + Drive mount
!nvidia-smi -L
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CACHE = "/content/drive/MyDrive/THGS_cache"
os.makedirs(DRIVE_CACHE, exist_ok=True)
print(f"Drive 캐시: {DRIVE_CACHE} -> {os.listdir(DRIVE_CACHE) if os.listdir(DRIVE_CACHE) else '비어있음'}")

In [ ]:
# THGS repo 클론 + 최신 pull
import os
REPO_URL = "https://github.com/BAEJUNHAK/THGS.git"
WORK_DIR = "/content/THGS"
if not os.path.exists(WORK_DIR):
    !git clone --recursive {REPO_URL} {WORK_DIR}
os.chdir(WORK_DIR)
!git pull --rebase --autostash 2>&1 | tail -3
print("work dir:", os.getcwd())

In [ ]:
# 의존성 — A 베이스라인과 동일 + ultralytics(FastSAM)
!pip install -q --upgrade opencv-python
!pip install -q plyfile==0.8.1 omegaconf open_clip_torch transformers \
    pycocotools scikit-image scikit-learn einops natsort gdown \
    seaborn matplotlib tqdm pandas
# FastSAM
!pip install -q ultralytics
# segment-anything (LangSplat fork) — A 인코딩 호환용 (이번 노트북에선 안 쓰지만 소스 의존성)
!pip install -q git+https://github.com/minghanqin/segment-anything-langsplat.git || true

import open_clip, ultralytics
print("open_clip:", open_clip.__version__, "| ultralytics:", ultralytics.__version__)

---
## Cell 1. 데이터셋 + A의 language_features 복원

베이스라인이 만든 SAM 4-scale features 를 그대로 재사용 (재인코딩 안 함).
GT 라벨 폴더(`data/lerf-ovs/label/ramen`)도 같이 확보.

In [ ]:
import os, shutil, zipfile, gdown
from omegaconf import OmegaConf

DATASET = "lerf"
CONFIG_FILE = f"configs/{DATASET}.yml"
SCENE = "ramen"
cfg = OmegaConf.load(CONFIG_FILE)
DATA_PATH = cfg.dataset.data_path  # data/lerf-ovs

# LERF-OVS 데이터 (이미지 + label JSON) 다운로드
LERF_DATA_ID = "1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt"
LERF_DATA_ZIP = "/content/lerf_ovs.zip"
if not os.path.exists(f"{DATA_PATH}/{SCENE}/images"):
    print("LERF-OVS 다운로드 ...")
    gdown.download(id=LERF_DATA_ID, output=LERF_DATA_ZIP, quiet=False)
    with zipfile.ZipFile(LERF_DATA_ZIP, 'r') as z:
        z.extractall("data/")
    os.remove(LERF_DATA_ZIP)
    if not os.path.exists(DATA_PATH) and os.path.exists("data/lerf_ovs"):
        os.symlink("lerf_ovs", DATA_PATH)
print(f"images: {os.path.exists(f'{DATA_PATH}/{SCENE}/images')}")
print(f"labels: {os.path.exists(f'{DATA_PATH}/label/{SCENE}')}")

In [ ]:
# A의 language_features 를 Drive 캐시에서 복원
local_feat_A = f"{DATA_PATH}/{SCENE}/language_features"
drive_feat_A = f"{DRIVE_CACHE}/{SCENE}_language_features"

if os.path.exists(local_feat_A) and len(os.listdir(local_feat_A)) > 0:
    print(f"[A feat] 로컬에 존재: {len(os.listdir(local_feat_A))} files")
elif os.path.exists(drive_feat_A):
    print(f"[A feat] Drive에서 복원 중...")
    shutil.copytree(drive_feat_A, local_feat_A)
    print(f"  복원 완료: {len(os.listdir(local_feat_A))} files")
else:
    raise RuntimeError(
        f"A의 language_features 캐시 없음. THGS_Pipeline_ramen.ipynb를 먼저 돌려 Drive에 백업하세요.\n"
        f"기대 위치: {drive_feat_A}"
    )

---
## Cell 2. FastSAM 가중치 준비

In [ ]:
import os, shutil
FASTSAM_CKPT = "ckpts/FastSAM-x.pt"
os.makedirs("ckpts", exist_ok=True)

if not os.path.exists(FASTSAM_CKPT):
    # 1차: ultralytics 자동 다운로드 (가장 신뢰성 높음)
    try:
        from ultralytics import FastSAM
        print("ultralytics 자동 다운로드 시도...")
        m = FastSAM('FastSAM-x.pt')   # 첫 호출 시 ultralytics CDN/GitHub assets에서 받음
        # ultralytics는 cwd 또는 ckpt_path 에 저장
        candidates = [
            'FastSAM-x.pt',
            getattr(m, 'ckpt_path', None),
            os.path.expanduser('~/.config/Ultralytics/FastSAM-x.pt'),
        ]
        for c in candidates:
            if c and os.path.exists(c):
                shutil.copy2(c, FASTSAM_CKPT)
                print(f"  복사: {c} -> {FASTSAM_CKPT}")
                break
        del m
    except Exception as e:
        print(f"ultralytics 자동 다운로드 실패: {e}")

# 2차 fallback: ultralytics assets 직접 wget
if not os.path.exists(FASTSAM_CKPT):
    print("fallback: GitHub assets 직접 다운로드")
    !wget -q --show-progress -O {FASTSAM_CKPT} \
        https://github.com/ultralytics/assets/releases/download/v8.2.0/FastSAM-x.pt \
        || wget -q --show-progress -O {FASTSAM_CKPT} \
        https://github.com/CASIA-IVA-Lab/FastSAM/releases/download/v0.0.1/FastSAM-x.pt

assert os.path.exists(FASTSAM_CKPT) and os.path.getsize(FASTSAM_CKPT) > 10_000_000, \
    f"FastSAM 가중치 다운로드 실패. 수동 확인: {FASTSAM_CKPT}"
print(f"FastSAM ckpt: {FASTSAM_CKPT} ({os.path.getsize(FASTSAM_CKPT)/1024/1024:.1f} MB)")

---
## Cell 3. B용 FastSAM+CLIP 인코딩

이미지 131장 전체 → `data/lerf-ovs/ramen/language_features_fastsam/<frame>_s.npy + _f.npy` 생성.
T4 GPU 기준 FastSAM-x 4-pass 면 이미지당 1~3초, 131장 전체 5~10분 예상.

동시에 timing CSV 도 출력 (`timing_fastsam.csv`).
Drive 백업까지 끝내면 다음 런타임에서 재실행 안 해도 됨.

In [ ]:
import os, shutil
local_feat_B = f"{DATA_PATH}/{SCENE}/language_features_fastsam"
drive_feat_B = f"{DRIVE_CACHE}/{SCENE}_language_features_fastsam"

# 0) Drive 캐시 우선 복원
if os.path.exists(local_feat_B) and len(os.listdir(local_feat_B)) >= 200:
    print(f"[B feat] 로컬에 이미 존재: {len(os.listdir(local_feat_B))} files")
elif os.path.exists(drive_feat_B):
    print(f"[B feat] Drive에서 복원 중...")
    if os.path.exists(local_feat_B):
        shutil.rmtree(local_feat_B)
    shutil.copytree(drive_feat_B, local_feat_B)
    if os.path.exists(f"{drive_feat_B}/../{SCENE}_timing_fastsam.csv"):
        shutil.copy2(f"{drive_feat_B}/../{SCENE}_timing_fastsam.csv",
                     f"{DATA_PATH}/{SCENE}/timing_fastsam.csv")
    print(f"  복원 완료: {len(os.listdir(local_feat_B))} files")
else:
    print("[B feat] 캐시 없음 - 신규 인코딩")
    !python scripts/encode_fastsam_clip.py \
        --source_path {DATA_PATH}/{SCENE} \
        --fastsam_ckpt {FASTSAM_CKPT} \
        --feat_subdir language_features_fastsam

In [ ]:
# B 산출물 + timing 을 Drive에 백업
if os.path.exists(local_feat_B) and not os.path.exists(drive_feat_B):
    print("Drive에 B 산출물 백업 중...")
    shutil.copytree(local_feat_B, drive_feat_B)
    timing_src = f"{DATA_PATH}/{SCENE}/timing_fastsam.csv"
    if os.path.exists(timing_src):
        shutil.copy2(timing_src, f"{DRIVE_CACHE}/{SCENE}_timing_fastsam.csv")
    print("백업 완료")
elif os.path.exists(drive_feat_B):
    print("Drive 백업 이미 존재")

---
## Cell 4. Mask proposer 품질 격리 (CLIP-free)

쿼리·CLIP 무관 — 각 GT polygon에 대해 "4-scale 마스크 중 가장 잘 맞는 mask의 IoU"만 측정.
A vs B 의 mask proposer 자체 품질 차이가 어느 정도인지를 격리해서 본다.

In [ ]:
EVAL_FRAMES = [
    "frame_00006", "frame_00024", "frame_00060", "frame_00065",
    "frame_00081", "frame_00119", "frame_00128",
]
OUT_ROOT = "output/exp_2d_source"
os.makedirs(OUT_ROOT, exist_ok=True)

# A: proposer quality only (skip metrics by giving empty queries — but we need queries for the script)
# Trick: run with full queries; proposer CSV is independent of CLIP.
# To save time, evaluate only on EVAL_FRAMES (--eval_frames).
QUERIES_JSON = "scripts/queries_ramen.json"
GT_DIR = f"{DATA_PATH}/label/{SCENE}"
IMG_DIR = f"{DATA_PATH}/{SCENE}/images"

print("=" * 60)
print("Mask proposer quality (CLIP-free) — A")
print("=" * 60)
!python scripts/eval_2d_source.py \
    --feat_dir {local_feat_A} --gt_dir {GT_DIR} --img_dir {IMG_DIR} \
    --queries_json {QUERIES_JSON} \
    --out_csv {OUT_ROOT}/_tmp_metrics_A_proposer.csv \
    --proposer_csv {OUT_ROOT}/proposer_A.csv \
    --arm A --scale_mode s_only \
    --eval_frames {' '.join(EVAL_FRAMES)}

print("\n" + "=" * 60)
print("Mask proposer quality (CLIP-free) — B")
print("=" * 60)
!python scripts/eval_2d_source.py \
    --feat_dir {local_feat_B} --gt_dir {GT_DIR} --img_dir {IMG_DIR} \
    --queries_json {QUERIES_JSON} \
    --out_csv {OUT_ROOT}/_tmp_metrics_B_proposer.csv \
    --proposer_csv {OUT_ROOT}/proposer_B.csv \
    --arm B --scale_mode s_only \
    --eval_frames {' '.join(EVAL_FRAMES)}

In [ ]:
# Proposer 품질 비교 표
import pandas as pd
pa = pd.read_csv(f"{OUT_ROOT}/proposer_A.csv")
pb = pd.read_csv(f"{OUT_ROOT}/proposer_B.csv")

# 각 (frame, category) 별 4-level 중 best level 의 IoU
def per_gt_best(df):
    return df.groupby(['frame','category'])['best_iou'].max().reset_index()

ba = per_gt_best(pa).rename(columns={'best_iou':'iou_A'})
bb = per_gt_best(pb).rename(columns={'best_iou':'iou_B'})
merged = ba.merge(bb, on=['frame','category'], how='outer')
merged['delta'] = merged['iou_B'] - merged['iou_A']
print("per (frame, category) best mask IoU:")
print(merged.round(3).to_string(index=False))
print("\n=== category 평균 ===")
print(merged.groupby('category').agg(iou_A=('iou_A','mean'),
                                       iou_B=('iou_B','mean'),
                                       delta=('delta','mean')).round(3).to_string())
print("\n=== 전체 평균 ===")
print(f"A: {merged['iou_A'].mean():.4f} | B: {merged['iou_B'].mean():.4f} | Δ(B-A): {merged['delta'].mean():+.4f}")

---
## Cell 5. A 평가 (s_only + best4)

In [ ]:
for mode in ['s_only', 'best4']:
    out_csv = f"{OUT_ROOT}/metrics_A_{mode}.csv"
    print(f"\n--- A / {mode} ---")
    !python scripts/eval_2d_source.py \
        --feat_dir {local_feat_A} --gt_dir {GT_DIR} --img_dir {IMG_DIR} \
        --queries_json {QUERIES_JSON} \
        --out_csv {out_csv} \
        --arm A --scale_mode {mode} \
        --eval_frames {' '.join(EVAL_FRAMES)}

---
## Cell 6. B 평가 (s_only + best4)

In [ ]:
for mode in ['s_only', 'best4']:
    out_csv = f"{OUT_ROOT}/metrics_B_{mode}.csv"
    print(f"\n--- B / {mode} ---")
    !python scripts/eval_2d_source.py \
        --feat_dir {local_feat_B} --gt_dir {GT_DIR} --img_dir {IMG_DIR} \
        --queries_json {QUERIES_JSON} \
        --out_csv {out_csv} \
        --arm B --scale_mode {mode} \
        --eval_frames {' '.join(EVAL_FRAMES)}

---
## Cell 7. Timing 통합 표

In [ ]:
import pandas as pd, os

timing_B_path = f"{DATA_PATH}/{SCENE}/timing_fastsam.csv"
tb = None
tb_sub = None
if not os.path.exists(timing_B_path):
    print(f"[WARN] {timing_B_path} 없음 - timing 측정값 누락")
else:
    tb = pd.read_csv(timing_B_path)
    # warm-up 5장 제외, 첫 30장 평균 (총 30장 미만이면 가용한 것 다 사용)
    if len(tb) > 5:
        tb_sub = tb.iloc[5:min(35, len(tb))]
    else:
        tb_sub = tb
    print(f"=== B (FastSAM-x + OpenCLIP ViT-B/16, 4-conf) timing ===")
    print(f"frames sampled (after 5-warmup): {len(tb_sub)}")
    print(f"  t_mask : {tb_sub['t_mask_ms'].mean():.1f} ± {tb_sub['t_mask_ms'].std():.1f} ms")
    print(f"  t_clip : {tb_sub['t_clip_ms'].mean():.1f} ± {tb_sub['t_clip_ms'].std():.1f} ms")
    print(f"  t_total: {tb_sub['t_total_ms'].mean():.1f} ± {tb_sub['t_total_ms'].std():.1f} ms")
    print(f"  n_masks: {tb_sub['n_masks'].mean():.1f}")
    print(f"  peak GPU mem: {tb_sub['peak_mem_mb'].mean():.0f} MB")

# A의 timing 은 베이스라인 인코딩에서 안 잼 (image_encoding.py 에 timing 없음).
# 참고치만: SAM ViT-H 4-scale auto-mask 는 보통 이미지당 10~30초 (T4 기준).
print("\n=== A (SAM ViT-H 4-scale + OpenCLIP) timing ===")
print("  (image_encoding.py 가 timing 안 잼. 일반 보고치: ~20-30s/img on T4)")
if tb_sub is not None and len(tb_sub) > 0:
    ratio = 25000.0 / tb_sub['t_total_ms'].mean()
    print(f"\n속도 ratio 추정: A/B ≈ 25000ms / {tb_sub['t_total_ms'].mean():.0f}ms")
    print(f"  → 약 {ratio:.0f}× 빠름 (B가)")

# 디스크 산출물 크기
def dir_size(p):
    if not os.path.exists(p): return 0
    return sum(os.path.getsize(os.path.join(p, f)) for f in os.listdir(p))
size_A = dir_size(local_feat_A) / 1024 / 1024
size_B = dir_size(local_feat_B) / 1024 / 1024
print(f"\n=== 디스크 산출물 ===")
print(f"  A: {size_A:.1f} MB")
print(f"  B: {size_B:.1f} MB")

---
## Cell 8. Level별 정성 시각화 (A grid, B grid)

In [ ]:
import os, json, numpy as np, cv2
import matplotlib.pyplot as plt
import torch
import sys; sys.path.insert(0, '.')
from utils.vlm_utils import ClipSimMeasure

VIZ_DIR = f"{OUT_ROOT}/viz"
os.makedirs(VIZ_DIR, exist_ok=True)

queries = json.load(open(QUERIES_JSON))
vlm = ClipSimMeasure(); vlm.load_model()

def load_features(feat_dir, frame):
    s = np.load(f"{feat_dir}/{frame}_s.npy").astype(np.int64)
    f = np.load(f"{feat_dir}/{frame}_f.npy")
    return s, f

def gt_polygons_for(frame, target_hw):
    p = f"{GT_DIR}/{frame}.json"
    if not os.path.exists(p): return {}
    anno = json.load(open(p))
    H, W = target_hw
    out = {}
    for o in anno['objects']:
        m = np.zeros((anno['info']['height'], anno['info']['width']), np.uint8)
        cv2.fillPoly(m, [np.asarray(o['segmentation'], np.int32)], 1)
        if (anno['info']['height'], anno['info']['width']) != (H, W):
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
        out[o['category']] = np.maximum(out.get(o['category'], np.zeros((H,W), np.uint8)), m)
    return out

def render_sim(feat_dir, frame, prompt, scale_mode='s_only'):
    seg, fmat = load_features(feat_dir, frame)
    fmat_t = torch.from_numpy(fmat).float().cuda()
    vlm.encode_text(prompt)
    with torch.no_grad():
        sim = vlm.compute_similarity(fmat_t).cpu().numpy()
    if scale_mode == 's_only':
        sm = seg[1]
        out = np.where(sm >= 0, sim[np.clip(sm, 0, len(sim)-1)], 0.0)
    else:  # best4
        out = np.zeros(seg.shape[1:], dtype=np.float32)
        for j in range(4):
            sm = seg[j]
            sj = np.where(sm >= 0, sim[np.clip(sm, 0, len(sim)-1)], 0.0)
            out = np.maximum(out, sj)
    return out, seg.shape[1:]

def viz_arm_level(arm_name, feat_dir, level_key, scale_mode='s_only', dpi=85):
    items = queries[level_key]
    R, C = len(items), len(EVAL_FRAMES)
    fig, axes = plt.subplots(R, C, figsize=(2.0*C, 1.8*R))
    if R == 1: axes = axes[None, :]
    fig.suptitle(f"{arm_name} / {level_key} / {scale_mode}", fontsize=11)
    for i, it in enumerate(items):
        for j, frm in enumerate(EVAL_FRAMES):
            try:
                sim, hw = render_sim(feat_dir, frm, it['q'], scale_mode)
            except Exception as e:
                axes[i,j].text(0.5,0.5,f'err',ha='center',va='center'); axes[i,j].axis('off'); continue
            img = cv2.imread(f"{IMG_DIR}/{frm}.jpg")
            img = cv2.resize(img, (hw[1], hw[0]))
            heat = (plt.cm.jet(np.clip(sim, 0, 1))[..., :3] * 255).astype(np.uint8)
            blend = (0.5*cv2.cvtColor(img, cv2.COLOR_BGR2RGB) + 0.5*heat).astype(np.uint8)
            # GT contour
            if it.get('gt'):
                gts = gt_polygons_for(frm, hw)
                if it['gt'] in gts:
                    cnts, _ = cv2.findContours(gts[it['gt']], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(blend, cnts, -1, (0,255,0), 2)
            axes[i,j].imshow(blend)
            axes[i,j].set_xticks([]); axes[i,j].set_yticks([])
            if i == 0:
                axes[i,j].set_title(frm.replace('frame_','f'), fontsize=8)
            if j == 0:
                lab = it['q'][:30] + ('…' if len(it['q'])>30 else '')
                axes[i,0].set_ylabel(lab, fontsize=7, rotation=0, ha='right', va='center', labelpad=80)
    plt.tight_layout()
    out_p = f"{VIZ_DIR}/{arm_name}_{level_key}_{scale_mode}.png"
    plt.savefig(out_p, dpi=dpi, bbox_inches='tight')
    plt.show()
    print("saved:", out_p)

# 모든 arm × level (L0/L1/L2/L5) — s_only 만 (best4는 보너스)
for arm_name, fd in [('A', local_feat_A), ('B', local_feat_B)]:
    for lvl in ['L0','L1','L2','L5']:
        viz_arm_level(arm_name, fd, lvl, 's_only')

---
## Cell 9. A vs B 비교 시각화 (선별 케이스)

L1 / L2 의 인상적 케이스 골라서 같은 (frame, query) 의 A vs B 를 나란히.

In [ ]:
PICKS = [
    ('L0', 'bowl', 'frame_00006'),
    ('L0', 'kamaboko', 'frame_00060'),
    ('L1', 'yellow bowl', 'frame_00006'),
    ('L2', 'egg yolk', 'frame_00006'),
    ('L2', 'egg white', 'frame_00006'),
    ('L2', 'noodle strand', 'frame_00060'),
    ('L5', 'A pair of utensils for holding food next to a yellow bowl.', 'frame_00006'),
]

fig, axes = plt.subplots(len(PICKS), 4, figsize=(14, 3.0*len(PICKS)))
if len(PICKS) == 1: axes = axes[None, :]
for i, (lvl, q, frm) in enumerate(PICKS):
    img = cv2.imread(f"{IMG_DIR}/{frm}.jpg")
    sim_a, hw = render_sim(local_feat_A, frm, q, 's_only')
    sim_b, _ = render_sim(local_feat_B, frm, q, 's_only')
    img = cv2.resize(img, (hw[1], hw[0]))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    axes[i,0].imshow(img_rgb); axes[i,0].set_title(f"{lvl}: {q[:35]}{'…' if len(q)>35 else ''} / {frm}", fontsize=9)
    axes[i,1].imshow(sim_a, cmap='jet', vmin=0, vmax=1); axes[i,1].set_title(f"A sim (max={sim_a.max():.2f})", fontsize=9)
    axes[i,2].imshow(sim_b, cmap='jet', vmin=0, vmax=1); axes[i,2].set_title(f"B sim (max={sim_b.max():.2f})", fontsize=9)
    diff = sim_b - sim_a
    axes[i,3].imshow(diff, cmap='RdBu_r', vmin=-0.5, vmax=0.5); axes[i,3].set_title(f"B-A diff", fontsize=9)
    for j in range(4): axes[i,j].axis('off')
plt.tight_layout()
out_p = f"{VIZ_DIR}/AB_picks.png"
plt.savefig(out_p, dpi=110, bbox_inches='tight')
plt.show()
print("saved:", out_p)

---
## Cell 10. 종합 차트 + summary.md

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

dfs = {}
for arm in ['A','B']:
    for mode in ['s_only','best4']:
        p = f"{OUT_ROOT}/metrics_{arm}_{mode}.csv"
        if os.path.exists(p):
            d = pd.read_csv(p)
            d['key'] = f"{arm}_{mode}"
            dfs[f"{arm}_{mode}"] = d
df = pd.concat(dfs.values(), ignore_index=True)
df_q = df[df['has_gt']==True]   # 정량 가능만

# (1) Level × arm mIoU
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ax = axes[0,0]
piv = df_q.groupby(['level','key'])['IoU'].mean().unstack('key')
piv = piv.reindex(['L0','L1','L5'])
piv.plot(kind='bar', ax=ax)
ax.set_title('mIoU per Level (GT 있는 Level만)'); ax.set_ylabel('mIoU'); ax.set_xticklabels(piv.index, rotation=0)
for i, lvl in enumerate(piv.index):
    for j, col in enumerate(piv.columns):
        v = piv.loc[lvl, col]
        if not np.isnan(v):
            ax.text(i + j*0.2 - 0.3, v+0.005, f'{v:.2f}', fontsize=7)

# (2) Boundary F-score (BF@2)
ax = axes[0,1]
piv2 = df_q.groupby(['level','key'])['BF2'].mean().unstack('key')
piv2 = piv2.reindex(['L0','L1','L5'])
piv2.plot(kind='bar', ax=ax)
ax.set_title('Boundary F-score @ 2px'); ax.set_ylabel('BF@2'); ax.set_xticklabels(piv2.index, rotation=0)

# (3) PR-AUC (threshold-free)
ax = axes[1,0]
piv3 = df_q.groupby(['level','key'])['PR_AUC'].mean().unstack('key')
piv3 = piv3.reindex(['L0','L1','L5'])
piv3.plot(kind='bar', ax=ax)
ax.set_title('mAP / PR-AUC (threshold-free)'); ax.set_ylabel('PR-AUC'); ax.set_xticklabels(piv3.index, rotation=0)

# (4) Speed-Quality scatter (s_only 기준)
ax = axes[1,1]
# tb_sub는 Cell 7에서 계산됨. 재실행 안전성을 위해 None 가드.
try:
    _tb_sub = tb_sub
except NameError:
    _tb_sub = None
if _tb_sub is not None and len(_tb_sub) > 0:
    t_b = _tb_sub['t_total_ms'].mean()
else:
    t_b = float('nan')
t_a = 25000.0  # 추정
miou_a = df_q[df_q['key']=='A_s_only']['IoU'].mean()
miou_b = df_q[df_q['key']=='B_s_only']['IoU'].mean()
ax.scatter([t_a], [miou_a], s=200, label=f'A (SAM ViT-H, est. {t_a/1000:.0f}s)', color='C0')
if not np.isnan(t_b):
    ax.scatter([t_b], [miou_b], s=200, label=f'B (FastSAM, {t_b:.0f}ms)', color='C1')
ax.set_xscale('log'); ax.set_xlabel('time per image (ms, log)'); ax.set_ylabel('mIoU')
ax.set_title('Speed–Quality (s_only, all-level avg)')
ax.legend()

plt.tight_layout()
summary_png = f"{OUT_ROOT}/summary.png"
plt.savefig(summary_png, dpi=130, bbox_inches='tight')
plt.show()
print("saved:", summary_png)

---
## Cell 11. Phase 1.5 — 텍스트 인코더 무관 평가 (피처 클러스터링)

같은 무대에서 A/B 를 비교하기 위해 **텍스트 쿼리를 거치지 않고** "피처가 인스턴스를 얼마나 잘 분리하는가" 만 측정한다.
나중에 C(OpenSeg)/D(Mask-Adapter) 처럼 텍스트 인코더가 다른 arm 을 추가해도 이 단계는 그대로 비교 가능하다.

**메트릭**:
- ARI (Adjusted Rand Index): 클러스터링 일치도
- NMI (Normalized Mutual Information): 정보량 기준 일치도
- Cluster Purity: 각 cluster 가 단일 GT category 에 얼마나 집중됐나
- Hungarian-matched mIoU: cluster ↔ GT category 최적 매칭 후 픽셀 IoU 평균


In [ ]:
# A / B 둘 다 cluster eval 실행 (s level = 1)
import os
GT_DIR = f"{DATA_PATH}/label/{SCENE}"

for arm, fdir in [('A', local_feat_A), ('B', local_feat_B)]:
    out_csv = f"{OUT_ROOT}/cluster_{arm}.csv"
    viz_dir = f"{OUT_ROOT}/cluster_viz_{arm}"
    print("=" * 60)
    print(f"Cluster quality (text-encoder-agnostic) — {arm}")
    print("=" * 60)
    !python scripts/eval_cluster_quality.py \
        --feat_dir {fdir} --gt_dir {GT_DIR} \
        --out_csv {out_csv} --viz_dir {viz_dir} \
        --arm {arm} --level 1 --min_iou 0.2 \
        --eval_frames {' '.join(EVAL_FRAMES)}


In [ ]:
# 결과 집계 + 막대 차트
import pandas as pd, os, numpy as np, matplotlib.pyplot as plt

cluster_dfs = {}
for arm in ['A', 'B']:
    p = f"{OUT_ROOT}/cluster_{arm}.csv"
    if os.path.exists(p):
        d = pd.read_csv(p)
        d['arm'] = arm
        cluster_dfs[arm] = d

if len(cluster_dfs) == 0:
    print("[cluster] no CSV found")
else:
    cdf = pd.concat(cluster_dfs.values(), ignore_index=True)
    print("[cluster] frame-level rows:", len(cdf))

    summary_cluster = cdf.groupby('arm')[['ARI', 'NMI', 'purity', 'hung_mIoU',
                                          'mean_assignment_iou']].mean()
    print("\n=== Mean over frames ===")
    print(summary_cluster.round(4))

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, metric, title in zip(
        axes,
        ['ARI', 'NMI', 'purity', 'hung_mIoU'],
        ['ARI ↑', 'NMI ↑', 'Cluster Purity ↑', 'Hungarian mIoU ↑']
    ):
        v = summary_cluster[metric]
        ax.bar(v.index, v.values, color=['C0', 'C1'][:len(v)])
        for i, val in enumerate(v.values):
            ax.text(i, val + 0.01, f'{val:.3f}', ha='center', fontsize=10)
        ax.set_title(title); ax.set_ylim(0, max(1.0, v.max() * 1.15))
    plt.tight_layout()
    cluster_png = f"{OUT_ROOT}/cluster_summary.png"
    plt.savefig(cluster_png, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"saved: {cluster_png}")

    # 정성 그림 미리보기 (A 1장 + B 1장)
    sample_frame = EVAL_FRAMES[0]
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for col, arm in enumerate(['A', 'B']):
        for row, suffix in enumerate(['pixfeat_pca', 'mask_tsne']):
            p = f"{OUT_ROOT}/cluster_viz_{arm}/{sample_frame}_{suffix}.png"
            ax = axes[row, col]
            if os.path.exists(p):
                ax.imshow(plt.imread(p))
                ax.set_title(f"{arm} — {suffix} ({sample_frame})", fontsize=9)
            else:
                ax.text(0.5, 0.5, f'(missing)\n{p}', ha='center', va='center',
                        transform=ax.transAxes, fontsize=8)
                ax.set_title(f"{arm} — {suffix}", fontsize=9)
            ax.axis('off')
    plt.tight_layout()
    plt.savefig(f"{OUT_ROOT}/cluster_viz_preview.png", dpi=110, bbox_inches='tight')
    plt.show()


In [ ]:
# summary.md 자동 생성
lines = []
lines.append("# THGS 2D-Source Ablation — Phase 1 Summary (ramen)\n")
lines.append("## Setup\n")
lines.append("- A: SAM ViT-H 4-scale auto-mask + OpenCLIP ViT-B/16 224-crop (D=512)\n")
lines.append("- B: FastSAM-x 4-conf-thresh [0.4,0.5,0.6,0.7] + OpenCLIP ViT-B/16 224-crop (D=512)\n")
lines.append(f"- Eval frames: {len(EVAL_FRAMES)} ({', '.join(EVAL_FRAMES)})\n")
lines.append(f"- Queries: L0={len(queries['L0'])}, L1={len(queries['L1'])}, L2={len(queries['L2'])}, L5={len(queries['L5'])}\n")
lines.append("\n## Quality (mIoU per level, s_only / best4)\n")
lines.append("| Level | A_s_only | A_best4 | B_s_only | B_best4 |\n")
lines.append("|---|---|---|---|---|\n")
for lvl in ['L0','L1','L5']:
    row = [lvl]
    for k in ['A_s_only','A_best4','B_s_only','B_best4']:
        d = df_q[(df_q['level']==lvl) & (df_q['key']==k)]['IoU']
        row.append(f"{d.mean():.4f}" if len(d) else "-")
    lines.append("| " + " | ".join(row) + " |\n")
lines.append("\n## Boundary F-score @ 2px\n")
lines.append("| Level | A_s_only | B_s_only |\n|---|---|---|\n")
for lvl in ['L0','L1','L5']:
    a = df_q[(df_q['level']==lvl)&(df_q['key']=='A_s_only')]['BF2'].mean()
    b = df_q[(df_q['level']==lvl)&(df_q['key']=='B_s_only')]['BF2'].mean()
    lines.append(f"| {lvl} | {a:.4f} | {b:.4f} |\n")
lines.append("\n## Speed\n")
lines.append("- A (estimated): ~25 s / image (T4, SAM ViT-H 4-scale)\n")
try:
    _tb_sub = tb_sub
except NameError:
    _tb_sub = None
if _tb_sub is not None and len(_tb_sub) > 0:
    t_b_ms = _tb_sub['t_total_ms'].mean()
    lines.append(f"- B (measured) : {t_b_ms:.0f} ms / image (T4)\n")
    lines.append(f"- Speedup     : ~{25000/t_b_ms:.0f}× (B faster)\n")
else:
    lines.append("- B (measured) : timing CSV 없음\n")
lines.append("\n## Mask proposer quality (CLIP-free, mean best-mask IoU vs GT)\n")
try:
    _merged = merged
    lines.append(f"- A: {_merged['iou_A'].mean():.4f}\n")
    lines.append(f"- B: {_merged['iou_B'].mean():.4f}\n")
    lines.append(f"- Δ : {_merged['delta'].mean():+.4f} (B − A)\n")
except NameError:
    lines.append("- (Cell 4의 merged DataFrame 없음 — 재실행 필요)\n")


# Cluster eval (text-encoder-agnostic)
try:
    _cluster_csv_a = f"{OUT_ROOT}/cluster_A.csv"
    _cluster_csv_b = f"{OUT_ROOT}/cluster_B.csv"
    if os.path.exists(_cluster_csv_a) and os.path.exists(_cluster_csv_b):
        import pandas as pd
        _ca = pd.read_csv(_cluster_csv_a); _cb = pd.read_csv(_cluster_csv_b)
        lines.append("\n## Phase 1.5 — Text-encoder-agnostic clustering (s level)\n")
        lines.append("| arm | ARI | NMI | Purity | Hung-mIoU | mean_assign_IoU |\n")
        lines.append("|-----|-----|-----|--------|-----------|-----------------|\n")
        for arm, _cd in [('A', _ca), ('B', _cb)]:
            lines.append(
                f"| {arm} | {_cd['ARI'].mean():.3f} | {_cd['NMI'].mean():.3f} | "
                f"{_cd['purity'].mean():.3f} | {_cd['hung_mIoU'].mean():.3f} | "
                f"{_cd['mean_assignment_iou'].mean():.3f} |\n"
            )
        lines.append("\nSee: `cluster_summary.png`, `cluster_viz_A/`, `cluster_viz_B/`\n")
except Exception as _e:
    print("[summary.md] cluster section skipped:", _e)

with open(f"{OUT_ROOT}/summary.md", 'w') as f:
    f.writelines(lines)
print(''.join(lines))
print("\nsaved:", f"{OUT_ROOT}/summary.md")

In [ ]:
# 결과 폴더 전체를 Drive에 백업
import shutil, os
drive_out = f"{DRIVE_CACHE}/{SCENE}_exp_2d_source"
if os.path.exists(drive_out):
    shutil.rmtree(drive_out)
shutil.copytree(OUT_ROOT, drive_out)
print(f"backed up to {drive_out}")
print("contents:", os.listdir(drive_out))